In [2]:
# google_play_crawler.py

from google_play_scraper import app, reviews, Sort
import pandas as pd
import time
import os

os.makedirs("output/csv", exist_ok=True)

# 各銀行 APP 的 Package ID
BANK_APPS = {
    "國泰世華": "com.cathaybk.mymobibank.android",
    "台北富邦": "com.fubon.mobilebank",
    "中國信託": "com.ctbc.mymobibank",
    "玉山銀行": "com.esun.financial",
    "台新銀行": "com.taishin.mymobibank",
    "永豐銀行": "tw.com.sinopac.mobilebank",
}


# ── 功能一：抓取 APP 基本評分資訊 ───────────────────────────
def get_app_info(package_id: str, bank_name: str) -> dict:
    """
    抓取單一 APP 的評分、評論數、版本等基本資訊
    """
    try:
        result = app(
            package_id,
            lang="zh_TW",   # 語言
            country="tw"    # 國家：台灣
        )

        return {
            "bank_name":    bank_name,
            "package_id":   package_id,
            "app_name":     result["title"],
            "rating":       result["score"],           # 評分（1~5）
            "rating_count": result["ratings"],         # 評分人數
            "review_count": result["reviews"],         # 評論數
            "installs":     result["installs"],        # 下載次數（區間值）
            "version":      result["version"],         # 最新版本
            "update_date":  result["updated"],         # 最後更新時間
            "size":         result["size"],            # APP 大小
            "developer":    result["developer"],       # 開發者
            "scraped_date": pd.Timestamp.today().strftime("%Y-%m-%d"),
        }

    except Exception as e:
        print(f"❌ {bank_name} 基本資訊抓取失敗：{e}")
        return {"bank_name": bank_name, "package_id": package_id, "error": str(e)}


# ── 功能二：抓取用戶評論內容 ────────────────────────────────
def get_app_reviews(package_id: str, bank_name: str, count: int = 100) -> pd.DataFrame:
    """
    抓取用戶評論（最多可抓幾千則）
    count: 要抓幾則評論
    """
    try:
        result, _ = reviews(
            package_id,
            lang="zh_TW",
            country="tw",
            sort=Sort.NEWEST,    # 最新評論優先
            count=count
        )

        records = []
        for r in result:
            records.append({
                "bank_name":    bank_name,
                "review_id":    r["reviewId"],
                "username":     r["userName"],
                "rating":       r["score"],          # 該評論的評分（1~5）
                "content":      r["content"],        # 評論內容
                "reply":        r["replyContent"],   # 官方回覆
                "thumbs_up":    r["thumbsUpCount"],  # 有用數
                "review_date":  r["at"].strftime("%Y-%m-%d"),
                "app_version":  r["reviewCreatedVersion"],
            })

        df = pd.DataFrame(records)
        print(f"✅ {bank_name}：取得 {len(df)} 則評論")
        return df

    except Exception as e:
        print(f"❌ {bank_name} 評論抓取失敗：{e}")
        return pd.DataFrame()


# ── 功能三：計算月度評分趨勢 ────────────────────────────────
def calc_monthly_rating(reviews_df: pd.DataFrame) -> pd.DataFrame:
    """
    從評論資料計算每月平均評分（模擬趨勢）
    """
    reviews_df["review_date"] = pd.to_datetime(reviews_df["review_date"])
    reviews_df["year_month"] = reviews_df["review_date"].dt.to_period("M")

    monthly = reviews_df.groupby(["bank_name", "year_month"]).agg(
        avg_rating   = ("rating", "mean"),
        review_count = ("rating", "count"),
        one_star     = ("rating", lambda x: (x == 1).sum()),
        five_star    = ("rating", lambda x: (x == 5).sum()),
    ).reset_index()

    monthly["avg_rating"] = monthly["avg_rating"].round(2)
    return monthly


# ── 主程式 ───────────────────────────────────────────────────
def crawl_all_apps():
    all_info    = []
    all_reviews = []

    for bank_name, pkg_id in BANK_APPS.items():
        print(f"\n📱 爬取 {bank_name}...")

        # 基本資訊
        info = get_app_info(pkg_id, bank_name)
        all_info.append(info)
        print(f"   評分：{info.get('rating')} / 評論數：{info.get('review_count')}")

        # 評論內容（每家抓 200 則）
        rev_df = get_app_reviews(pkg_id, bank_name, count=200)
        all_reviews.append(rev_df)

        time.sleep(2)   # 避免請求過快

    # 儲存基本資訊
    info_df = pd.DataFrame(all_info)
    info_df.to_csv("output/csv/app_ratings.csv",
                   index=False, encoding="utf-8-sig")
    print("\n✅ 基本資訊已儲存：output/csv/app_ratings.csv")

    # 儲存所有評論
    reviews_df = pd.concat(all_reviews, ignore_index=True)
    reviews_df.to_csv("output/csv/app_reviews.csv",
                      index=False, encoding="utf-8-sig")
    print("✅ 評論資料已儲存：output/csv/app_reviews.csv")

    # 計算月度趨勢
    monthly_df = calc_monthly_rating(reviews_df)
    monthly_df.to_csv("output/csv/app_monthly_rating.csv",
                      index=False, encoding="utf-8-sig")
    print("✅ 月度評分趨勢已儲存：output/csv/app_monthly_rating.csv")

    return info_df, reviews_df, monthly_df


if __name__ == "__main__":
    info_df, reviews_df, monthly_df = crawl_all_apps()

    print("\n📊 各銀行 APP 評分總覽：")
    print(info_df[["bank_name", "rating", "review_count", "version", "update_date"]])

    print("\n📅 月度評分趨勢（最新3個月）：")
    print(monthly_df.tail(18))


📱 爬取 國泰世華...
❌ 國泰世華 基本資訊抓取失敗：'size'
   評分：None / 評論數：None
✅ 國泰世華：取得 200 則評論

📱 爬取 台北富邦...
❌ 台北富邦 基本資訊抓取失敗：App not found(404).
   評分：None / 評論數：None
✅ 台北富邦：取得 0 則評論

📱 爬取 中國信託...
❌ 中國信託 基本資訊抓取失敗：App not found(404).
   評分：None / 評論數：None
✅ 中國信託：取得 0 則評論

📱 爬取 玉山銀行...
❌ 玉山銀行 基本資訊抓取失敗：App not found(404).
   評分：None / 評論數：None
✅ 玉山銀行：取得 0 則評論

📱 爬取 台新銀行...
❌ 台新銀行 基本資訊抓取失敗：App not found(404).
   評分：None / 評論數：None
✅ 台新銀行：取得 0 則評論

📱 爬取 永豐銀行...
❌ 永豐銀行 基本資訊抓取失敗：App not found(404).
   評分：None / 評論數：None
✅ 永豐銀行：取得 0 則評論

✅ 基本資訊已儲存：output/csv/app_ratings.csv
✅ 評論資料已儲存：output/csv/app_reviews.csv
✅ 月度評分趨勢已儲存：output/csv/app_monthly_rating.csv

📊 各銀行 APP 評分總覽：


KeyError: "['rating', 'review_count', 'version', 'update_date'] not in index"